# SAR flood mapping & before/after water change — Cameroon (Far North / Logone)

The **default**, auth-free workflow: pull Sentinel-1 GRD backscatter for a **pre-flood** and a **post-flood** date over the Logone-et-Chari floodplain in Cameroon's Far North from the open **Earth Search** STAC catalogue (no account, no sign-in), convert to decibels, Otsu-threshold each scene to a water mask, compute the before/after flood extent, quantify the flooded hectares, map the flood, and validate the estimate against an official UN OCHA flood situation report for the 2024 event.

**Inspired by** [`robmarkcole/satellite-image-deep-learning`](https://github.com/robmarkcole/satellite-image-deep-learning) — the field's best catalogue of EO problem ideas, where flood mapping is one of the scoped problems. The design choice here: flood mapping done right uses **Sentinel-1 SAR**, which sees through cloud (optical sensors are blinded exactly when it floods), with classic **Otsu** automatic thresholding on backscatter. The runnable, tested contribution is the pure-numpy flood core (`floodmap.threshold` / `floodmap.water` / `floodmap.change`) — the same code the unit tests exercise — and this notebook is the documented real Sentinel-1 pull.

## 0. Setup

The STAC path needs the geospatial stack. The pure-numpy core does not.

```bash
conda env create -f environment.yml && conda activate flood-change-detection
# or: pip install -r requirements.txt
```

On Colab: `!pip install pystac-client odc-stac rioxarray rasterio matplotlib pyyaml` and clone the repo so the `floodmap` package is importable (`%pip install -e .` from the project folder, or add `src` to `sys.path`).

In [ ]:
# %pip install pystac-client odc-stac rioxarray rasterio matplotlib pyyaml
import matplotlib.pyplot as plt
import numpy as np
import yaml

## 1. Configuration

All analysis choices come from `config/aoi.yaml`: the Logone-et-Chari bbox, the pre-flood and post-flood dates, the SAR polarization (VH for open water), the orbit, and the pixel size.

In [ ]:
with open("../config/aoi.yaml") as fh:
    cfg = yaml.safe_load(fh)

b = cfg["aoi"]["bbox"]
bbox = [b["min_lon"], b["min_lat"], b["max_lon"], b["max_lat"]]
a = cfg["analysis"]
pre_date = cfg["pre_date"]
post_date = cfg["post_date"]
bbox, pre_date, post_date, a["polarization"], a["orbit"]

## 2. Pull Sentinel-1 GRD pre/post backscatter from STAC

`floodmap.stac.build_flood_inputs` searches Earth Search for Sentinel-1 GRD items over the AOI for each date, on a single orbit direction so the incidence angle is consistent, loads the VH band, mosaics same-day passes, and returns the backscatter for each date **in decibels** — the scale Otsu thresholding expects. Auth-free.

CLI equivalent: `floodmap map --config config/aoi.yaml`.

In [ ]:
from floodmap.stac import build_flood_inputs

pre_db, post_db = build_flood_inputs(
    bbox,
    pre_date=pre_date,
    post_date=post_date,
    orbit=a["orbit"],
    polarization=a["polarization"].lower(),
    resolution=a["pixel_size_m"],
)
pre_db.shape, post_db.shape

## 3. Otsu water threshold per date

Over a partly flooded scene the dB-backscatter histogram is **bimodal**: a low (dark) water peak and a higher land peak. `floodmap.threshold.otsu_threshold` finds the between-class-variance-maximising split in the valley between them — automatically, with no hand-tuned cut-off. We threshold each date independently, since the post scene has a much larger water population.

In [ ]:
from floodmap.threshold import otsu_threshold

pre_thresh = otsu_threshold(pre_db)
post_thresh = otsu_threshold(post_db)
print(f"pre  Otsu water threshold: {pre_thresh:6.2f} dB")
print(f"post Otsu water threshold: {post_thresh:6.2f} dB")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for axis, db, t, label in [(ax[0], pre_db, pre_thresh, "pre"), (ax[1], post_db, post_thresh, "post")]:
    axis.hist(db[np.isfinite(db)].ravel(), bins=128, color="0.6")
    axis.axvline(t, color="C3", lw=2, label=f"Otsu = {t:.1f} dB")
    axis.set_title(f"{label} VH backscatter (dB)")
    axis.set_xlabel("dB")
    axis.legend()
plt.tight_layout()
plt.show()

## 4. Water masks and before/after flood extent (the pure-numpy core)

This is the tested contribution. SAR water is **dark**, so water is *below* the threshold (`polarity="below"`). `flood_extent` splits the two masks into **flooded** (post & ~pre — the new flood), **permanent_water** (the channel/lake, water at both dates), and **receded**. `flood_stats` converts pixel counts to hectares.

In [ ]:
from floodmap.change import flood_extent, flood_stats
from floodmap.water import water_mask

pre_water = water_mask(pre_db, pre_thresh, polarity="below")
post_water = water_mask(post_db, post_thresh, polarity="below")

masks = flood_extent(pre_water, post_water)
stats = flood_stats(masks, a["pixel_size_m"])
stats

## 5. Map the flooded area

Pre water (permanent channel), post water (channel + flood), and the flood-extent classification: newly **flooded** (red), **permanent water** (blue), **receded** (orange).

In [ ]:
rgb = np.zeros((*post_water.shape, 3))
rgb[masks["permanent_water"]] = [0.1, 0.3, 0.9]  # blue
rgb[masks["flooded"]] = [0.9, 0.1, 0.1]          # red
rgb[masks["receded"]] = [1.0, 0.6, 0.0]          # orange

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(pre_water, cmap="Blues")
ax[0].set_title("Pre water (permanent)")
ax[1].imshow(post_water, cmap="Blues")
ax[1].set_title("Post water (channel + flood)")
ax[2].imshow(rgb)
ax[2].set_title("Flooded (red) / permanent (blue) / receded (orange)")
for x in ax:
    x.axis("off")
plt.tight_layout()
plt.show()

## 6. Validate against the official OCHA flood report

The credibility check: compare our Sentinel-1 flooded-area estimate against an **independent, official** figure for the same event. For the **2024 Far North floods**, UN OCHA situation reports (via [ReliefWeb](https://reliefweb.int/disaster/fl-2024-000162-cmr)) recorded extensive farmland inundation — **~82,509 hectares of farmland flooded** region-wide, with **Logone-et-Chari ~156,000 people affected** (OCHA, 3 October 2024).

Two caveats make this an order-of-magnitude check rather than a like-for-like match: (1) the OCHA farmland figure is region-wide, while our bbox covers only part of the Logone-et-Chari floodplain, so scale our estimate to the same footprint before comparing; (2) OCHA counts *farmland* damaged, while SAR maps *all* standing water (including non-agricultural land and excluding flooded-then-drained areas). Agreement within a reasonable margin supports the SAR total; a large gap points at speckle, permanent/flood-water confusion, or AOI mismatch (see Limitations).

In [ ]:
# Official figures for the 2024 Far North floods (UN OCHA, via ReliefWeb).
ocha_farmland_flooded_ha_region = 82_509   # region-wide farmland flooded, OCHA Oct 2024
ocha_people_affected_logone = 156_000      # Logone-et-Chari people affected, OCHA 3 Oct 2024

sar_flooded_ha = stats["flooded_hectares"]
print(f"Sentinel-1 SAR new-flood extent (this AOI): {sar_flooded_ha:,.0f} ha")
print(f"OCHA farmland flooded (Far North region)  : {ocha_farmland_flooded_ha_region:,.0f} ha")
print(f"OCHA people affected (Logone-et-Chari)    : {ocha_people_affected_logone:,.0f}")
print()
print("Compare like-for-like by clipping the OCHA footprint to this bbox, or by")
print("expanding the AOI to the full Logone-et-Chari floodplain (see config/aoi.yaml).")

## 7. Limitations

- **Speckle.** SAR's multiplicative speckle scatters isolated misclassified pixels; a smoothing / morphological clean-up before thresholding reduces it.
- **Permanent vs flood water.** The before/after design removes the channel, but a pre scene with its own high water (wet season "pre") undercounts the flood; choose a genuinely dry pre date.
- **Layover & radar shadow.** Over terrain, layover and shadow create dark patches that mimic water; the flat Logone floodplain is favourable, but hilly AOIs need a DEM mask.
- **Single orbit / incidence angle.** Backscatter depends on incidence angle and orbit; keep both fixed across pre and post (done here) so the comparison is fair.
- **Optical fallback is cloud-blocked.** The Sentinel-2 MNDWI fallback (`floodmap.water.mndwi`) only works on cloud-free days — usually unavailable during a flood, which is the whole reason SAR is the default.
- **Validation needed.** OCHA counts damaged farmland, not standing-water extent; treat the comparison as an order-of-magnitude sanity check, and confirm with a higher-resolution or field reference before reporting a number.